In [1]:
# =========================================
# 0. CREATE OUTPUT FOLDER
# =========================================
import os

output_folder = "commentsComparison"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)


In [2]:

# =========================================
# 1. IMPORT LIBRARIES
# =========================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import warnings
warnings.filterwarnings("ignore")

In [3]:
# =========================================
# 2. LOAD COMMENTS DATA
# =========================================
comments = pd.read_csv("Data/English_version/English_translated_comment_data.csv")
df = comments

print("Dataset shape:", df.shape)


Dataset shape: (257497, 10)


In [4]:
# =========================================
# 3. TEXT CLEANING
# =========================================
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df['clean'] = df['translated_text'].apply(clean_text)
df = df[df['clean'] != ""]


In [5]:
# =========================================
# 4. SENTIMENT ANALYSIS
# =========================================
def get_sentiment(text):
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    if polarity > 0.1:
        sentiment = "Positive"
    elif polarity < -0.1:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    return polarity, subjectivity, sentiment

df[['polarity','subjectivity','sentiment']] = df['clean'].apply(
    lambda x: pd.Series(get_sentiment(x))
)


In [6]:
# =========================================
# 5. SENTIMENT SUMMARY
# =========================================
sent_count = df['sentiment'].value_counts()
sent_percent = (sent_count / len(df)) * 100

print("\nSentiment Count:\n", sent_count)
print("\nSentiment Percentage:\n", sent_percent)


# =========================================
# 6. SENTIMENT GRAPH
# =========================================
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='sentiment')
plt.title("Sentiment Distribution (Comments)")

plt.savefig(f"{output_folder}/sentiment_distribution.png")
plt.close()




Sentiment Count:
 sentiment
Neutral     179827
Positive     62306
Negative     14081
Name: count, dtype: int64

Sentiment Percentage:
 sentiment
Neutral     70.186251
Positive    24.317953
Negative     5.495796
Name: count, dtype: float64


In [7]:
# =========================================
# 7. POLARITY DISTRIBUTION
# =========================================
plt.figure(figsize=(6,4))
sns.histplot(df['polarity'], kde=True)
plt.title("Polarity Distribution")

plt.savefig(f"{output_folder}/polarity_distribution.png")
plt.close()


In [8]:
# =========================================
# 8. TOPIC MODELING
# =========================================
vectorizer = CountVectorizer(stop_words='english', min_df=2)
dtm = vectorizer.fit_transform(df['clean'])

lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(dtm)

feature_names = vectorizer.get_feature_names_out()

print("\nTop Words per Topic:")
for i, topic in enumerate(lda.components_):
    words = [feature_names[j] for j in topic.argsort()[-10:]]
    print(f"Topic {i}: {words}")




Top Words per Topic:
Topic 0: ['love', 'popper', 'party', 'tantrik', 'da', 'mir', 'taranath', 'loading', 'red', 'heart']
Topic 1: ['joy', 'crying', 'hands', 'tears', 'suspense', 'sunday', 'eyes', 'story', 'smiling', 'face']
Topic 2: ['amar', 'folded', 'golpokathak', 'priyabrata', 'ta', 'na', 'hands', 'er', 'golpo', 'kore']


In [9]:
# =========================================
# 9. ASSIGN TOPICS
# =========================================
doc_topics = lda.transform(dtm)
df['Topic_Number'] = np.argmax(doc_topics, axis=1)

topic_labels = {
    0: "Positive Feedback",
    1: "Requests/Suggestions",
    2: "Emotional Response"
}

df['Topic_Label'] = df['Topic_Number'].map(topic_labels)



In [10]:
# =========================================
# 10. TOPIC SUMMARY
# =========================================
topic_count = df['Topic_Label'].value_counts()
topic_percent = (topic_count / len(df)) * 100

print("\nTopic Count:\n", topic_count)
print("\nTopic Percentage:\n", topic_percent)



Topic Count:
 Topic_Label
Requests/Suggestions    93254
Positive Feedback       90174
Emotional Response      72786
Name: count, dtype: int64

Topic Percentage:
 Topic_Label
Requests/Suggestions    36.396918
Positive Feedback       35.194798
Emotional Response      28.408284
Name: count, dtype: float64


In [11]:
# =========================================
# 11. TOPIC GRAPH
# =========================================
plt.figure(figsize=(6,4))
topic_count.plot(kind='bar')
plt.title("Topic Distribution")

plt.savefig(f"{output_folder}/topic_distribution.png")
plt.close()


In [12]:
# =========================================
# 12. SAVE OUTPUT FILES
# =========================================
df.to_csv(f"{output_folder}/comments_full_analysis.csv", index=False)

sent_count.to_csv(f"{output_folder}/sentiment_count.csv")
sent_percent.to_csv(f"{output_folder}/sentiment_percentage.csv")

topic_count.to_csv(f"{output_folder}/topic_count.csv")
topic_percent.to_csv(f"{output_folder}/topic_percentage.csv")

print("\n✅ All files saved in 'commentsComparison' folder!")


✅ All files saved in 'commentsComparison' folder!
